# Notebook 01 — Exploratory Data Analysis
**Project**: Multilingual Indian Sentiment Analysis (IndicSenti)  
**Palette**: mint=#F1F6F4, gold=#FFC801, teal=#114C5A, sage=#D9E8E2, orange=#FF9932, dark=#172B36

In [ ]:
# ── Imports & palette setup ─────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

# Design palette
PALETTE = {
    'mint':   '#F1F6F4',
    'gold':   '#FFC801',
    'teal':   '#114C5A',
    'sage':   '#D9E8E2',
    'orange': '#FF9932',
    'dark':   '#172B36',
}
SENTIMENT_COLORS = [PALETTE['teal'], PALETTE['gold'], PALETTE['orange']]
LANG_COLORS = [PALETTE['teal'], PALETTE['gold'], PALETTE['orange'],
               PALETTE['sage'], '#7CB9C4', '#F9A825']

# Apply palette globally
plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'],
    'axes.facecolor':   PALETTE['mint'],
    'axes.edgecolor':   PALETTE['teal'],
    'axes.labelcolor':  PALETTE['dark'],
    'text.color':       PALETTE['dark'],
    'xtick.color':      PALETTE['dark'],
    'ytick.color':      PALETTE['dark'],
    'grid.color':       PALETTE['sage'],
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  PALETTE['teal'],
    'figure.dpi':       120,
})
sns.set_palette(SENTIMENT_COLORS)
print('Palette configured.')

In [ ]:
# ── Load dataset ────────────────────────────────────────────
DATA_DIR = Path('../data')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f'Total samples: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Fig 1: Class distribution ────────────────────────────────
LABEL_MAP = {0: 'Positive', 1: 'Neutral', 2: 'Negative'}
df['sentiment_name'] = df['label'].map(LABEL_MAP)

counts = df['sentiment_name'].value_counts().reindex(['Positive', 'Neutral', 'Negative'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 1 — Overall Sentiment Distribution', fontsize=14,
             color=PALETTE['teal'], fontweight='bold')

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=SENTIMENT_COLORS, edgecolor='white', width=0.6)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{v:,}', ha='center', fontweight='bold', color=PALETTE['dark'])
axes[0].set_title('Sample Counts')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.4)
axes[0].spines[['top', 'right']].set_visible(False)

# Donut
wedge_props = dict(width=0.45, edgecolor='white', linewidth=2)
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=SENTIMENT_COLORS,
    autopct='%1.1f%%', wedgeprops=wedge_props, startangle=90
)
for t in autotexts: t.set_color('white'); t.set_fontweight('bold')
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.savefig('../reports/fig1_class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Per-language breakdown ───────────────────────────
LANGUAGES = ['Hindi', 'Tamil', 'Bengali', 'Marathi', 'Telugu', 'English']

lang_sent = df.groupby(['language', 'sentiment_name']).size().unstack(fill_value=0)
lang_sent = lang_sent.reindex(LANGUAGES)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(LANGUAGES))
width = 0.25

for i, (sent, color) in enumerate(zip(['Positive', 'Neutral', 'Negative'], SENTIMENT_COLORS)):
    bars = ax.bar(x + i*width, lang_sent[sent], width, label=sent,
                  color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(LANGUAGES)
ax.set_ylabel('Sample Count')
ax.set_title('Figure 2 — Sentiment Distribution by Language')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/fig2_per_language_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Text length distribution ─────────────────────────
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')

df['token_count'] = df['text'].apply(
    lambda t: len(tokenizer.encode(str(t), truncation=False))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 3 — Token Length Analysis', color=PALETTE['teal'], fontweight='bold')

# Overall histogram
axes[0].hist(df['token_count'].clip(0, 200), bins=40,
             color=PALETTE['teal'], edgecolor='white', alpha=0.85)
axes[0].axvline(128, color=PALETTE['orange'], linestyle='--', linewidth=2,
                label='Truncation (128)')
axes[0].axvline(df['token_count'].median(), color=PALETTE['gold'], linewidth=2,
                label=f'Median ({df["token_count"].median():.0f})')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Frequency')
axes[0].set_title('All Languages')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# Boxplot by language
lang_order = LANGUAGES
lang_data  = [df[df['language'] == l]['token_count'].clip(0,200).values for l in lang_order]
bp = axes[1].boxplot(lang_data, labels=lang_order, patch_artist=True, widths=0.5,
                     medianprops=dict(color=PALETTE['gold'], linewidth=2))
for patch, color in zip(bp['boxes'], LANG_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1].set_ylabel('Tokens'); axes[1].set_title('By Language')
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

# Stats table
stats = df.groupby('language')['token_count'].agg(['mean', 'median', 'max']).round(1)
print('\nToken Length Statistics:\n', stats)
print(f'\nTruncation rate (>128 tokens): {(df["token_count"] > 128).mean()*100:.1f}%')

plt.tight_layout()
plt.savefig('../reports/fig3_token_lengths.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Top tokens per sentiment ─────────────────────────
from collections import Counter
import re

def top_tokens(texts, n=20):
    tokens = []
    for t in texts:
        tokens.extend(re.findall(r'\b[a-zA-Z\u0900-\u097F\u0B80-\u0BFF\u0980-\u09FF]{3,}\b',
                                  str(t).lower()))
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 4 — Top 15 Tokens by Sentiment Class', color=PALETTE['teal'], fontweight='bold')

for ax, (label, color) in zip(axes, zip(['Positive', 'Neutral', 'Negative'], SENTIMENT_COLORS)):
    subset = df[df['sentiment_name'] == label]['text']
    top    = top_tokens(subset, n=15)
    words, freqs = zip(*top)
    ax.barh(list(reversed(words)), list(reversed(freqs)), color=color, edgecolor='white')
    ax.set_title(label)
    ax.set_xlabel('Frequency')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/fig4_top_tokens.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 5: Brand distribution ────────────────────────────────
brand_counts = df['brand'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(brand_counts.index, brand_counts.values,
               color=PALETTE['teal'], edgecolor='white', alpha=0.85)
for bar in bars:
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,}', va='center', fontsize=9, color=PALETTE['dark'])
ax.set_xlabel('Sample Count')
ax.set_title('Figure 5 — Samples per Brand', color=PALETTE['teal'])
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/fig5_brand_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary statistics table ─────────────────────────────────
summary = pd.DataFrame({
    'Total Samples': [len(df)],
    'Languages': [df['language'].nunique()],
    'Brands': [df['brand'].nunique()],
    'Avg Tokens': [df['token_count'].mean().round(1)],
    'Median Tokens': [df['token_count'].median()],
    'Truncation Rate': [(df['token_count'] > 128).mean().round(3)],
    'Class Balance (pos/neu/neg)': [
        '/'.join((df['sentiment_name'].value_counts(normalize=True)*100).round(1).astype(str).values)
    ],
})
summary.T.rename(columns={0: 'Value'})